# Conv Autoencoder for Background Filtering
Train a conv autoencoder on background windows, then keep high-MSE windows for clustering.
This notebook uses the month_split CSVs generated by analysis/make_month_splits.py.

## 1) Setup

In [ ]:
import csv
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

project_root = Path(".").resolve()
if not (project_root / "utilities").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from utilities import configs, feature_utils as futils

def set_seeds(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 2) Configuration
Update these paths and knobs as needed.

In [ ]:
mode = "inference"  # "train" or "inference"
run_name = "tracy_towseyn0.5"
project_root = Path(".").resolve()
if not (project_root / "utilities").exists():
    project_root = project_root.parent
results_dir = project_root / "analysis" / "autoencoder_results" / run_name
month_split_dir = results_dir / "month_split"
npz_root = project_root / "data" / "TracyNpz"

pretrained_checkpoint = project_root / "analysis" / "autoencoder_results" / run_name / "autoencoder_model.pt"

train_csv = month_split_dir / "train_ae_noise_pool.csv"
val_csv = month_split_dir / "val_evaluation_windows.csv"

# Threshold selection target
target_recall = 0.8

# Split hygiene
val_bg_exclusion_secs = 5.0

# Threshold sweep range (quantiles of train errors)
sweep_quantile_min = 0.0
sweep_quantile_max = 0.999
sweep_steps = 400

# Windowing
window_secs = 5.0
stride_secs = 3.0
mel_start = 9
mel_end = 128

# Training
epochs = 12
batch_size = 32
latent_dim = 128
learning_rate = 1e-3
seed = 42
max_train_windows = 60000
max_val_windows = 80000

# Scoring for clustering (optional)
# For inference, set score_csv to a CSV with columns: file,start_sec
score_csv = month_split_dir / "train_month_windows.csv"
score_out_csv = results_dir / "cluster_candidates.csv"
keep_only_anomalies = True

results_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 3) Load split CSVs and NPZ index

In [ ]:
set_seeds(seed)

if mode == "train":
    if not train_csv.exists() or not val_csv.exists():
        raise FileNotFoundError("Missing split CSVs. Run analysis/make_month_splits.py first.")

npz_files = sorted(npz_root.rglob("*.npz"))
if not npz_files:
    raise FileNotFoundError(f"No .npz files found under {npz_root}")
npz_lookup = {path.name: path for path in npz_files}

spec_cfg = configs.get_specgram_config()
secs_per_frame = spec_cfg["hop_length"] / spec_cfg["sample_rate"]
window_frames = max(1, round(window_secs / secs_per_frame))
stride_frames = max(1, round(stride_secs / secs_per_frame))

print(f"Window frames: {window_frames}, stride frames: {stride_frames}")

Window frames: 625, stride frames: 375


## 4) Build window records (train mode)
This loads windows into memory. Use max_train_windows/max_val_windows to cap size.

In [ ]:
def build_records(csv_path: Path, limit: int | None = None):
    df = pd.read_csv(csv_path)
    if limit is not None and len(df) > limit:
        df = df.sample(n=limit, random_state=seed).sort_values(["file", "start_sec"]).reset_index(drop=True)

    records = []
    n_files = df["file"].nunique()
    for file_name, group in tqdm(df.groupby("file", sort=False), desc=f"Loading {csv_path.name}", unit="file", total=n_files):
        npz_path = npz_lookup.get(file_name)
        if npz_path is None:
            continue

        spectrogram, _ = futils.load_spectrogram(npz_path, n_mels=spec_cfg["n_mels"], key="feature")
        for _, row in group.iterrows():
            start_sec = float(row["start_sec"])
            start_frame = int(round(start_sec / secs_per_frame))
            if start_frame + window_frames > spectrogram.shape[1]:
                continue

            window = spectrogram[mel_start:mel_end, start_frame:start_frame + window_frames]
            record = {
                "file": file_name,
                "source_path": str(npz_path),
                "start_frame": start_frame,
                "start_sec": start_sec,
                "window": window.astype(np.float32),
            }
            if "label" in df.columns:
                record["label"] = int(row["label"])
            records.append(record)

    return records

train_records = []
val_records = []
input_mean = None
input_std = None

if mode == "train":
    train_records = build_records(train_csv, limit=max_train_windows)
    val_records = build_records(val_csv, limit=max_val_windows)

    if not train_records or not val_records:
        raise RuntimeError("No train/val records loaded.")

    train_stack = np.stack([r["window"] for r in train_records], axis=0)
    input_mean = float(train_stack.mean())
    input_std = float(train_stack.std()) if float(train_stack.std()) > 0 else 1.0

    print(f"Train windows: {len(train_records)} | Val windows: {len(val_records)}")
else:
    print(f"Skipping train/val record build (mode={mode}).")

Skipping train/val record build (mode=inference).


## 5) Dataset + Model

In [ ]:
class WindowTensorDataset(Dataset):
    def __init__(self, items, mean, std):
        self.items = list(items)
        self.mean = float(mean)
        self.std = float(std) if float(std) > 0 else 1.0

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        x = torch.from_numpy(item["window"]).float()
        x = (x - self.mean) / self.std
        return x.unsqueeze(0), item["file"], float(item["start_sec"]), item["source_path"], int(item["start_frame"])

class ConvAutoencoder(nn.Module):
    def __init__(self, input_shape, latent_dim=128):
        super().__init__()
        self.input_shape = tuple(input_shape)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 8)),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, *self.input_shape)
            encoded = self.encoder(dummy)
            self.encoded_shape = tuple(encoded.shape[1:])
            encoded_dim = int(np.prod(self.encoded_shape))

        self.to_latent = nn.Sequential(
            nn.Flatten(),
            nn.Linear(encoded_dim, latent_dim),
            nn.ReLU(inplace=True),
        )
        self.from_latent = nn.Sequential(
            nn.Linear(latent_dim, encoded_dim),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(8, 1, kernel_size=3, padding=1),
        )

    def forward(self, x):
        x = self.encoder(x)
        z = self.to_latent(x)
        x = self.from_latent(z).view(-1, *self.encoded_shape)
        x = self.decoder(x)
        x = torch.nn.functional.interpolate(x, size=self.input_shape, mode="bilinear", align_corners=False)
        return x

def run_epoch(model, loader, optimizer=None, device="cpu"):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_items = 0
    criterion = nn.MSELoss()

    progress = tqdm(loader, desc="train" if training else "eval", leave=False)
    for batch in progress:
        x = batch[0].to(device)
        if training:
            optimizer.zero_grad()
        recon = model(x)
        loss = criterion(recon, x)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_items += x.size(0)
        progress.set_postfix(loss=f"{loss.item():.5f}")

    return total_loss / max(1, total_items)

## 6) Train (skipped when mode = "inference")

In [ ]:
if mode == "train":
    train_ds = WindowTensorDataset(train_records, input_mean, input_std)
    val_ds = WindowTensorDataset(val_records, input_mean, input_std)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

    input_shape = train_ds[0][0].shape[-2:]
    model = ConvAutoencoder(input_shape=input_shape, latent_dim=latent_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    history = []
    for epoch in tqdm(range(1, epochs + 1), desc="epochs"):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer, device=device)
        val_loss = run_epoch(model, val_loader, optimizer=None, device=device)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"Epoch {epoch:02d}/{epochs}  train={train_loss:.5f}  val={val_loss:.5f}")

    history_df = pd.DataFrame(history)
else:
    input_shape = None
    history_df = pd.DataFrame()
    print(f"Skipping training (mode={mode}).")

Skipping training (mode=inference).


In [ ]:
# Load pretrained checkpoint (inference mode)
if mode == "inference":
    if not pretrained_checkpoint.exists():
        raise FileNotFoundError(f"Checkpoint not found: {pretrained_checkpoint}")

    checkpoint = torch.load(pretrained_checkpoint, map_location=device)
    window_secs = float(checkpoint.get("window_secs", window_secs))
    stride_secs = float(checkpoint.get("stride_secs", stride_secs))
    mel_start = int(checkpoint.get("mel_start", mel_start))
    mel_end = int(checkpoint.get("mel_end", mel_end))
    window_frames = max(1, round(window_secs / secs_per_frame))
    stride_frames = max(1, round(stride_secs / secs_per_frame))

    input_mean = float(checkpoint["input_mean"])
    input_std = float(checkpoint["input_std"])
    threshold = float(checkpoint["threshold"])
    input_shape = tuple(checkpoint["input_shape"])
    latent_dim = int(checkpoint["latent_dim"])

    model = ConvAutoencoder(input_shape=input_shape, latent_dim=latent_dim).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded checkpoint: {pretrained_checkpoint}")
    print(f"Using threshold: {threshold:.6f}")
    print(f"Window frames: {window_frames}, stride frames: {stride_frames}")
else:
    print("Mode is train; skipping checkpoint load.")

C:\Users\Aleks\AppData\Local\Temp\ipykernel_12016\191692337.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained_checkpoint, map_location=d

Loaded checkpoint: D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\Tracy_towseyn0.5\autoencoder_model.pt
Using threshold: 0.637244
Window frames: 625, stride frames: 375


In [ ]:
# Inference: score all windows in NPZs and write MSEs
if mode == "inference":
    out_scores_csv = results_dir / "inference_window_scores.csv"
    out_scores_csv.parent.mkdir(parents=True, exist_ok=True)

    threshold_val = threshold if "threshold" in globals() else None
    fieldnames = ["file", "source_path", "start_sec", "start_frame", "reconstruction_mse"]
    if threshold_val is not None:
        fieldnames.append("is_anomaly")

    def _flush_batch(windows, meta, writer):
        x = torch.from_numpy(np.stack(windows, axis=0)).float().unsqueeze(1)
        x = (x - input_mean) / input_std
        x = x.to(device)
        with torch.no_grad():
            recon = model(x)
            errors = ((recon - x) ** 2).mean(dim=(1, 2, 3)).detach().cpu().numpy()

        for (file_name, source_path, start_sec, start_frame), error in zip(meta, errors):
            row = {
                "file": file_name,
                "source_path": source_path,
                "start_sec": float(start_sec),
                "start_frame": int(start_frame),
                "reconstruction_mse": float(error),
            }
            if threshold_val is not None:
                row["is_anomaly"] = int(error >= threshold_val)
            writer.writerow(row)

    with out_scores_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for npz_path in tqdm(npz_files, desc="Scoring NPZs", unit="file"):
            spectrogram, _ = futils.load_spectrogram(npz_path, n_mels=spec_cfg["n_mels"], key="feature")
            windows = []
            meta = []

            for start_frame, window in futils.windows_from_spectrogram(
                spectrogram,
                window_frames=window_frames,
                stride_frames=stride_frames,
                mel_start=mel_start,
                mel_end=mel_end,
            ):
                windows.append(window.astype(np.float32))
                start_sec = round(start_frame * secs_per_frame, 3)
                meta.append((npz_path.name, str(npz_path), start_sec, start_frame))

                if len(windows) >= batch_size:
                    _flush_batch(windows, meta, writer)
                    windows, meta = [], []

            if windows:
                _flush_batch(windows, meta, writer)

    print(f"Saved window scores to {out_scores_csv}")
else:
    print(f"Skipping inference scoring (mode={mode}).")

Scoring NPZs:   0%|          | 0/1939 [00:00<?, ?file/s]

Scoring NPZs: 100%|██████████| 1939/1939 [04:35<00:00,  7.05file/s]

Saved window scores to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\inference_window_scores.csv


## 7) Score + Threshold Selection (train mode)
Select threshold that maximizes recall while removing as much background as possible.

In [ ]:
def score_records(model, items, mean, std, device="cpu", batch_size=64):
    dataset = WindowTensorDataset(items, mean, std)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    rows = []
    model.eval()
    with torch.no_grad():
        for x, files, start_secs, source_paths, start_frames in tqdm(loader, desc="scoring", leave=False):
            x = x.to(device)
            recon = model(x)
            errors = ((recon - x) ** 2).mean(dim=(1, 2, 3)).detach().cpu().numpy()
            for i, error in enumerate(errors):
                rows.append(
                    {
                        "file": files[i],
                        "source_path": source_paths[i],
                        "start_sec": float(start_secs[i]),
                        "start_frame": int(start_frames[i]),
                        "reconstruction_mse": float(error),
                    }
                )
    return pd.DataFrame(rows)

def compute_binary_metrics(labels, preds):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=int)
    tp = int(((labels == 1) & (preds == 1)).sum())
    fp = int(((labels == 0) & (preds == 1)).sum())
    fn = int(((labels == 1) & (preds == 0)).sum())
    tn = int(((labels == 0) & (preds == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
    }

if mode == "train":
    train_scores = score_records(model, train_records, input_mean, input_std, device=device)
    val_scores = score_records(model, val_records, input_mean, input_std, device=device)

    val_meta = pd.DataFrame({
        "record_idx": np.arange(len(val_records), dtype=int),
        "label": [int(r.get("label", 0)) for r in val_records],
    })
    val_scores = val_scores.reset_index(drop=True)
    val_scores["record_idx"] = np.arange(len(val_scores), dtype=int)
    val_eval = val_scores.merge(val_meta, on="record_idx", how="left")

    if not (0.0 <= sweep_quantile_min < sweep_quantile_max <= 1.0):
        raise ValueError("sweep quantiles must satisfy 0 <= min < max <= 1")
    if sweep_steps < 2:
        raise ValueError("sweep_steps must be >= 2")

    candidate_thresholds = np.unique(
        np.quantile(
            train_scores["reconstruction_mse"],
            np.linspace(sweep_quantile_min, sweep_quantile_max, sweep_steps),
        )
    )

    metric_rows = []
    for candidate in tqdm(candidate_thresholds, desc="threshold sweep", unit="thr"):
        preds = (val_eval["reconstruction_mse"].to_numpy() >= candidate).astype(int)
        metrics = compute_binary_metrics(val_eval["label"].to_numpy(), preds)
        metrics["threshold"] = float(candidate)
        metric_rows.append(metrics)

    metrics_df = pd.DataFrame(metric_rows)
    metrics_df["bg_pass"] = metrics_df["fp"] / (metrics_df["fp"] + metrics_df["tn"])

    eligible = metrics_df[metrics_df["recall"] >= target_recall].copy()
    if eligible.empty:
        best = metrics_df.sort_values(["recall", "bg_pass"], ascending=[False, True]).iloc[0]
        selection_note = f"target recall {target_recall:.3f} not met; using max recall"
    else:
        best = eligible.sort_values(["bg_pass", "threshold"], ascending=[True, False]).iloc[0]
        selection_note = f"recall >= {target_recall:.3f}"

    threshold = float(best["threshold"])
    val_eval["is_anomaly"] = val_eval["reconstruction_mse"] >= threshold

    print(f"Selected threshold: {threshold:.6f} ({selection_note})")
    print(f"Precision={best['precision']:.4f} Recall={best['recall']:.4f} F1={best['f1']:.4f} Specificity={best['specificity']:.4f}")
    print(f"Background removed: {1.0 - best['bg_pass']:.3f}")

    train_scores.to_csv(results_dir / "train_scores.csv", index=False)
    val_eval.to_csv(results_dir / "val_scores_labeled.csv", index=False)
    metrics_df.to_csv(results_dir / "val_threshold_sweep.csv", index=False)

    checkpoint_path = results_dir / "autoencoder_model.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_shape": tuple(train_records[0]["window"].shape),
            "latent_dim": int(latent_dim),
            "input_mean": float(input_mean),
            "input_std": float(input_std),
            "threshold": float(threshold),
            "window_secs": float(window_secs),
            "stride_secs": float(stride_secs),
            "mel_start": int(mel_start),
            "mel_end": int(mel_end),
            "run_name": run_name,
            "npz_root": str(npz_root),
        },
        checkpoint_path,
    )

    (results_dir / "run_config.json").write_text(
        json.dumps(
            {
                "npz_root": str(npz_root),
                "train_csv": str(train_csv),
                "val_csv": str(val_csv),
                "epochs": epochs,
                "batch_size": batch_size,
                "latent_dim": latent_dim,
                "learning_rate": learning_rate,
                "max_train_windows": max_train_windows,
                "max_val_windows": max_val_windows,
                "seed": seed,
                "threshold": threshold,
                "target_recall": target_recall,
                "sweep_quantile_min": sweep_quantile_min,
                "sweep_quantile_max": sweep_quantile_max,
                "sweep_steps": sweep_steps,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    metrics_df = pd.DataFrame()
    val_eval = pd.DataFrame()
    print(f"Skipping threshold sweep (mode={mode}).")
    if "threshold" in globals():
        print(f"Using threshold: {threshold:.6f}")

Skipping threshold sweep (mode=inference).
Using threshold: 0.637244


## 7b) Latent dim sweep (train mode)
Compare smaller latent sizes and pick the best recall/background tradeoff.
Defaults use a subset and fewer epochs for speed.

In [ ]:
if mode == "train":
    latent_dim_grid = [32, 64, 128]
    sweep_epochs = 3
    sweep_train_limit = None  # set to None for full train set
    sweep_val_limit = None    # set to None for full val set
    sweep_batch_size = batch_size

    def _sample_records(records, limit, seed):
        if limit is None or len(records) <= limit:
            return records
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(records), size=limit, replace=False)
        return [records[i] for i in idx]

    def score_errors(model, records, mean, std, device="cpu", batch_size=64):
        dataset = WindowTensorDataset(records, mean, std)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
        errs = []
        model.eval()
        with torch.no_grad():
            for x, *_ in tqdm(loader, desc="scoring", leave=False):
                x = x.to(device)
                recon = model(x)
                batch_err = ((recon - x) ** 2).mean(dim=(1, 2, 3)).detach().cpu().numpy()
                errs.append(batch_err)
        return np.concatenate(errs) if errs else np.array([])

    train_subset = _sample_records(train_records, sweep_train_limit, seed)
    val_subset = _sample_records(val_records, sweep_val_limit, seed)
    val_labels = np.array([int(r.get("label", 0)) for r in val_subset], dtype=int)

    sweep_results = []
    for ld in latent_dim_grid:
        model = ConvAutoencoder(input_shape=input_shape, latent_dim=ld).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

        train_ds = WindowTensorDataset(train_subset, input_mean, input_std)
        train_loader = DataLoader(train_ds, batch_size=sweep_batch_size, shuffle=True, drop_last=False)
        for _ in range(1, sweep_epochs + 1):
            _ = run_epoch(model, train_loader, optimizer=optimizer, device=device)

        train_err = score_errors(model, train_subset, input_mean, input_std, device=device, batch_size=sweep_batch_size)
        val_err = score_errors(model, val_subset, input_mean, input_std, device=device, batch_size=sweep_batch_size)
        if train_err.size == 0 or val_err.size == 0:
            continue

        candidate_thresholds = np.unique(
            np.quantile(train_err, np.linspace(sweep_quantile_min, sweep_quantile_max, sweep_steps))
        )
        metric_rows = []
        for thr in candidate_thresholds:
            preds = (val_err >= thr).astype(int)
            metrics = compute_binary_metrics(val_labels, preds)
            metrics["threshold"] = float(thr)
            denom = metrics["fp"] + metrics["tn"]
            metrics["bg_pass"] = metrics["fp"] / denom if denom else 0.0
            metric_rows.append(metrics)

        metrics_df = pd.DataFrame(metric_rows)
        eligible = metrics_df[metrics_df["recall"] >= target_recall]
        if eligible.empty:
            best = metrics_df.sort_values(["recall", "bg_pass"], ascending=[False, True]).iloc[0]
            status = "max recall"
        else:
            best = eligible.sort_values(["bg_pass", "threshold"], ascending=[True, False]).iloc[0]
            status = "recall ok"

        sweep_results.append({
            "latent_dim": ld,
            "status": status,
            "threshold": float(best["threshold"]),
            "recall": float(best["recall"]),
            "precision": float(best["precision"]),
            "bg_removed": float(1.0 - best["bg_pass"]),
            "tp": int(best["tp"]),
            "fp": int(best["fp"]),
            "fn": int(best["fn"]),
            "tn": int(best["tn"]),
        })

    sweep_df = pd.DataFrame(sweep_results)
    sweep_df.sort_values(["recall", "bg_removed"], ascending=[False, False])
else:
    sweep_df = pd.DataFrame()
    print(f"Skipping latent dim sweep (mode={mode}).")

Skipping latent dim sweep (mode=inference).


## 8) Plots (train mode)

In [ ]:
if mode == "train":
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
    ax.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.set_title("Autoencoder training curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    plot_df = metrics_df.sort_values("threshold", ascending=True)
    ax.plot(plot_df["threshold"], plot_df["recall"], marker="o", markersize=2, linewidth=1, label="Recall")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Recall")
    ax.set_title("Recall vs Threshold")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    kept_windows = plot_df["tp"] + plot_df["fp"]
    ax2.plot(plot_df["threshold"], kept_windows, color="dimgray", linestyle="--", linewidth=1, label="Windows kept")
    ax2.set_ylabel("Windows kept")
    ax2.tick_params(axis="y", labelcolor="dimgray")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    if lines2:
        ax.legend(lines1 + lines2, labels1 + labels2, loc="best")
    else:
        ax.legend()

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    background = val_eval[val_eval["label"] == 0]["reconstruction_mse"]
    calls = val_eval[val_eval["label"] == 1]["reconstruction_mse"]
    ax.hist(background, bins=40, alpha=0.6, color="steelblue", label="Validation background")
    ax.axvline(threshold, color="crimson", linestyle="--", label="Selected threshold")
    ax.set_xlabel("Reconstruction MSE")
    ax.set_ylabel("Window count")
    ax.set_title("Validation reconstruction error by label")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Skipping plots (mode={mode}).")

Skipping plots (mode=inference).


## 9) Score full window list for clustering 
This streams windows from a CSV and writes high-MSE candidates.

In [ ]:
def score_csv_stream(csv_path: Path, out_csv: Path, mse_threshold: float, keep_only: bool = True, batch_size: int = 64):
    df = pd.read_csv(csv_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = ["file", "source_path", "start_sec", "start_frame", "reconstruction_mse", "is_anomaly"]
    with out_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        n_files = df["file"].nunique()
        for file_name, group in tqdm(df.groupby("file", sort=False), desc="Scoring CSV", unit="file", total=n_files):
            npz_path = npz_lookup.get(file_name)
            if npz_path is None:
                continue

            spectrogram, _ = futils.load_spectrogram(npz_path, n_mels=spec_cfg["n_mels"], key="feature")
            windows = []
            meta = []

            for _, row in group.iterrows():
                start_sec = float(row["start_sec"])
                start_frame = int(round(start_sec / secs_per_frame))
                if start_frame + window_frames > spectrogram.shape[1]:
                    continue
                window = spectrogram[mel_start:mel_end, start_frame:start_frame + window_frames]
                windows.append(window.astype(np.float32))
                meta.append((file_name, str(npz_path), start_sec, start_frame))

                if len(windows) >= batch_size:
                    _flush_batch(windows, meta, writer, mse_threshold, keep_only)
                    windows, meta = [], []

            if windows:
                _flush_batch(windows, meta, writer, mse_threshold, keep_only)

def _flush_batch(windows, meta, writer, mse_threshold, keep_only):
    x = torch.from_numpy(np.stack(windows, axis=0)).float().unsqueeze(1)
    x = (x - input_mean) / input_std
    x = x.to(device)
    with torch.no_grad():
        recon = model(x)
        errors = ((recon - x) ** 2).mean(dim=(1, 2, 3)).detach().cpu().numpy()

    for (file_name, source_path, start_sec, start_frame), error in zip(meta, errors):
        is_anomaly = bool(error >= mse_threshold)
        if keep_only and not is_anomaly:
            continue
        writer.writerow({
            "file": file_name,
            "source_path": source_path,
            "start_sec": float(start_sec),
            "start_frame": int(start_frame),
            "reconstruction_mse": float(error),
            "is_anomaly": int(is_anomaly),
        })

if score_csv is not None and score_csv.exists():
    score_csv_stream(score_csv, score_out_csv, threshold, keep_only=keep_only_anomalies, batch_size=batch_size)
    print(f"Saved scored windows to {score_out_csv}")
else:
    print("score_csv not found or not set; skipping scoring.")

score_csv not found or not set; skipping scoring.


In [ ]:
# Visual inspection: save top-K highest MSE windows as separate spectrogram PNGs
sanity_csv = None
if "out_scores_csv" in globals() and out_scores_csv.exists():
    sanity_csv = out_scores_csv
elif score_out_csv.exists():
    sanity_csv = score_out_csv
elif score_csv is not None and score_csv.exists():
    sanity_csv = score_csv

# number of top windows to save
top_k = 100

if sanity_csv is None:
    print("Visual inspection skipped: score CSV not found.")
else:
    df = pd.read_csv(sanity_csv)
    if "reconstruction_mse" in df.columns:
        top_df = df.sort_values("reconstruction_mse", ascending=False).head(top_k).reset_index(drop=True)
        cols = [c for c in ["file", "start_sec", "start_frame", "reconstruction_mse", "source_path"] if c in top_df.columns]
        print(f"Top {top_k} windows by MSE from {sanity_csv}:")
        display(top_df.loc[:, cols].reset_index(drop=True))

        from analysis.plot_spectrogram_windows import plot_window_grid

        for idx, row in top_df.iterrows():
            file_name = row.get("file")
            npz_path = Path(row.get("source_path")) if pd.notna(row.get("source_path")) else npz_lookup.get(file_name)
            if npz_path is None or not npz_path.exists():
                print(f"Skipping missing NPZ for {file_name}")
                continue

            if "start_frame" in row and pd.notna(row.get("start_frame")):
                start_frame = int(row.get("start_frame"))
            else:
                start_frame = int(round(float(row.get("start_sec")) / secs_per_frame))

            spectrogram, _ = futils.load_spectrogram(npz_path, n_mels=spec_cfg["n_mels"], key="feature")
            if start_frame + window_frames > spectrogram.shape[1]:
                print(f"Skipping window beyond end for {npz_path.name} at frame {start_frame}")
                continue

            window = spectrogram[mel_start:mel_end, start_frame:start_frame + window_frames]
            mse = float(row.get("reconstruction_mse"))

            out_path = results_dir / f"top_{idx+1:03d}_{npz_path.stem}_{start_frame}.png"
            plot_window_grid(
                windows=[(start_frame, window)],
                title=f"{npz_path.name}  start={start_frame}  mse={mse:.6f}",
                secs_per_frame=secs_per_frame,
                cols=1,
                output_path=out_path,
                mel_start=mel_start,
                cmap="viridis",
            )
            print(f"Saved spectrogram to {out_path}")
    else:
        print("reconstruction_mse column not present in scores CSV; cannot pick top windows.")

Top 100 windows by MSE from D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\inference_window_scores.csv:


,file,start_sec,start_frame,reconstruction_mse,source_path
0,6229.220914180000.npz,153.0,19125,16.846638,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
1,6229.220914180000.npz,156.0,19500,16.674973,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
2,6229.220914180000.npz,159.0,19875,16.530397,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
3,6229.220914180000.npz,162.0,20250,16.291275,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
4,6229.220914180000.npz,150.0,18750,16.251358,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
...,...,...,...,...,...
95,6229.220923182000.npz,117.0,14625,6.753320,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
96,6229.220923070000.npz,150.0,18750,6.751256,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
97,6229.220914180000.npz,111.0,13875,6.659355,D:\BSc\Detecting-Narwhals\processedDataNPZFile...
98,6229.220921024000.npz,0.0,0,6.536322,D:\BSc\Detecting-Narwhals\processedDataNPZFile...


Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_001_6229.220914180000_19125.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_002_6229.220914180000_19500.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_003_6229.220914180000_19875.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_004_6229.220914180000_20250.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_005_6229.220914180000_18750.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_006_6229.220914180000_18375.png
Saved spectrogram to D:\BSc\Detecting-Narwhals\analysis\autoencoder_results\towsey_n0.5_inference_Sep_6229\top_007_6229.220914180000_18000.png